# Step 4: Staff Detection — Inference GUI

Interactive interface for running staff detection inference on any video.

**Tasks answered:**
- Task 1: Identify which frames contain a staff member (detected via `staff_tag`)
- Task 2 (Bonus): Record and display the XY coordinates of the staff in each frame

**Usage:** Run the cell below, then open `http://127.0.0.1:7860` in your browser.

## 4.1 Install Dependencies

In [ ]:
%pip install gradio ultralytics -q

## 4.2 Launch Gradio Interface

In [ ]:
import cv2
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend required for Gradio
import matplotlib.pyplot as plt
import gradio as gr
from ultralytics import YOLO

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ── Inference function with Hysteresis Temporal Smoothing ─────────────────
def run_inference(video_path, model_path, conf_thresh, skip_frames, smooth_frames):

    # ── Validate inputs ───────────────────────────────
    if video_path is None:
        return None, None, "❌ Please upload a video."
    if not os.path.exists(model_path):
        return None, None, f"❌ Model not found: {model_path}"

    # ── Video setup ───────────────────────────────────
    cap          = cv2.VideoCapture(video_path)
    fps          = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out_video_path = os.path.join(OUTPUT_DIR, "output_annotated.mp4")
    fourcc         = cv2.VideoWriter_fourcc(*"avc1")  # H.264 for browser playback
    writer         = cv2.VideoWriter(out_video_path, fourcc, fps, (width, height))

    model       = YOLO(model_path)
    results_log = []
    frame_idx   = 0

    # ── Hysteresis / Debounce Configuration ───────────
    detection_counter    = 0
    no_detection_counter = 0
    REQUIRED_FRAMES      = int(smooth_frames)  # Frames needed to confirm presence
    TOLERANCE            = 2                   # Allowed blank frames before dropout
    
    last_known_detections = None               # Memory register to hold state

    # ── Inference loop ────────────────────────────────
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % skip_frames == 0:
            results    = model(frame, conf=conf_thresh, verbose=False)
            detections = results[0].boxes
            
            # ── State Machine Update ──────────────────
            if len(detections) > 0:
                detection_counter += 1
                no_detection_counter = 0
                last_known_detections = detections  # Cache the latest valid boxes
            else:
                no_detection_counter += 1
                if no_detection_counter >= TOLERANCE:
                    # Clear state only when missing frames exceed allowed tolerance
                    detection_counter     = 0
                    last_known_detections = None

            staff_detected = False

            # ── Render Gate using Cached Memory ───────
            if detection_counter >= REQUIRED_FRAMES and last_known_detections is not None:
                staff_detected = True
                
                for box in last_known_detections:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    conf_val        = float(box.conf[0])
                    cx              = (x1 + x2) / 2
                    cy              = (y1 + y2) / 2
                    timestamp       = frame_idx / fps

                    # Log validated or stable-cached coordinates to prevent telemetry drops
                    results_log.append({
                        "frame"      : frame_idx,
                        "timestamp"  : round(timestamp, 3),
                        "x"          : round(cx, 1),
                        "y"          : round(cy, 1),
                        "x1"         : round(x1, 1),
                        "y1"         : round(y1, 1),
                        "x2"         : round(x2, 1),
                        "y2"         : round(y2, 1),
                        "confidence" : round(conf_val, 4),
                    })

                    # Render bounding box
                    cv2.rectangle(frame,
                                  (int(x1), int(y1)),
                                  (int(x2), int(y2)),
                                  (0, 255, 0), 2)

                    # Render centre point
                    cv2.circle(frame, (int(cx), int(cy)), 5, (0, 0, 255), -1)

                    # Label: class + confidence
                    cv2.putText(frame, f"staff_tag  conf:{conf_val:.2f}",
                                (int(x1), int(y1) - 25),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)

                    # Label: XY coordinates
                    cv2.putText(frame, f"X:{cx:.0f}  Y:{cy:.0f}",
                                (int(x1), int(y1) - 8),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)

            # ── Frame status overlay (Reflects Debounced Decision) ──
            status_text  = f"Frame: {frame_idx}  |  {'STAFF PRESENT' if staff_detected else ''}"
            status_color = (0, 255, 0) if staff_detected else (200, 200, 200)
            cv2.putText(frame, status_text, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, status_color, 2)

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()

    # ── Build results dataframe ───────────────────────
    df = pd.DataFrame(results_log)

    if df.empty:
        return out_video_path, None, "⚠️ No verified staff detected under current thresholds."

    # ── Save CSV ──────────────────────────────────────
    csv_path = os.path.join(OUTPUT_DIR, "staff_detections.csv")
    df.to_csv(csv_path, index=False)

    # ── Summary text ──────────────────────────────────
    summary = (
        f"{'='*45}\n"
        f"INFERENCE SUMMARY (HYSTERESIS SMOOTHING ACTIVE)\n"
        f"{'='*45}\n"
        f"Video duration      : {total_frames/fps:.2f} seconds\n"
        f"Frames processed    : {total_frames // skip_frames}\n"
        f"Stable staff frames : {df['frame'].nunique()}\n"
        f"First appearance    : frame {df['frame'].min()} ({df['timestamp'].min():.2f}s)\n"
        f"Last appearance     : frame {df['frame'].max()} ({df['timestamp'].max():.2f}s)\n"
        f"Mean confidence     : {df['confidence'].mean():.4f}\n"
        f"Mean X position     : {df['x'].mean():.1f} px\n"
        f"Mean Y position     : {df['y'].mean():.1f} px\n"
        f"{'='*45}\n"
        f"CSV saved to        : {csv_path}"
    )

    # ── XY trajectory plots ───────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(df["timestamp"], df["x"],
                 label="X (horizontal)", color="royalblue", linewidth=2)
    axes[0].plot(df["timestamp"], df["y"],
                 label="Y (vertical)", color="tomato", linewidth=2, linestyle="--")
    axes[0].set_xlabel("Timestamp (seconds)")
    axes[0].set_ylabel("Coordinate (pixels)")
    axes[0].set_title("Staff Tag Position Over Time")
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    sc = axes[1].scatter(
        df["x"], df["y"],
        c=df["timestamp"], cmap="plasma",
        s=30, alpha=0.8
    )
    axes[1].plot(df["x"], df["y"], color="gray", linewidth=0.8, alpha=0.5)
    axes[1].set_xlim(0, width)
    axes[1].set_ylim(height, 0)  # Invert Y to match image coordinate system
    axes[1].set_xlabel("X (pixels)")
    axes[1].set_ylabel("Y (pixels)")
    axes[1].set_title("Staff Tag Trajectory (colour = time)")
    plt.colorbar(sc, ax=axes[1], label="Timestamp (seconds)")
    axes[1].grid(True, alpha=0.3)

    plt.suptitle("Staff Detection — XY Coordinate Analysis", fontsize=13)
    plt.tight_layout()

    traj_path = os.path.join(OUTPUT_DIR, "staff_trajectory.png")
    plt.savefig(traj_path, dpi=150, bbox_inches="tight")
    plt.close()

    return out_video_path, traj_path, summary


# ── Gradio UI ─────────────────────────────────────────────────────────────
with gr.Blocks(title="Staff Detection") as demo:

    gr.Markdown("# 🎯 Staff Detection — FootfallCam AI Evaluation")
    gr.Markdown("Upload a video and run inference using the fine-tuned YOLOv8n model with an adjustable hysteresis temporal filter.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Configuration")
            video_input = gr.Video(label="Upload Video")
            model_input = gr.Textbox(
                value = r"runs\detect\runs\staff_tag\weights\best.pt",
                label = "Model Path (best.pt)"
            )
            conf_slider = gr.Slider(
                minimum = 0.10, maximum = 0.95,
                value   = 0.50, step    = 0.05,
                label   = "Confidence Threshold"
            )
            skip_slider = gr.Slider(
                minimum = 1, maximum = 10,
                value   = 1, step    = 1,
                label   = "Skip Frames"
            )
            smooth_slider = gr.Slider(
                minimum = 1, maximum = 10,
                value   = 3, step    = 1,
                label   = "Smoothing Frames (Activation Activation Window)"
            )
            run_button = gr.Button("▶  Run Inference", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### 📹 Annotated Video")
            video_output = gr.Video(label="Output Video")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📊 XY Trajectory")
            plot_output = gr.Image(label="Staff Trajectory")
        with gr.Column():
            gr.Markdown("### 📋 Summary")
            summary_output = gr.Textbox(label="Results", lines=14)

    run_button.click(
        fn      = run_inference,
        inputs  = [video_input, model_input, conf_slider, skip_slider, smooth_slider],
        outputs = [video_output, plot_output, summary_output]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "c:\Users\USER\anaconda3\envs\new_imooc_ai\lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\USER\anaconda3\envs\new_imooc_ai\lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "c:\Users\USER\anaconda3\envs\new_imooc_ai\lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\USER\anaconda3\envs\new_imooc_ai\lib\asyncio\proactor_events.py", line